# 第7.3節：最適化問題

バージョン1.10.10で動作確認しています．

事前にインストールが必要なパッケージ
+ LinearAlgebra
+ Plots
+ LaTeXStrings
+ Optim

In [ ]:
using LinearAlgebra, Plots, LaTeXStrings

In [ ]:
# f(x) = (a - x1)^2 + b(x2 - x1^2)^2
function rosenbrock(x; a=1.0, b=100.0)
    x1, x2 = x
    return (a - x1)^2 + b*(x2 - x1^2)^2
end

function grad_rosenbrock(x; a=1.0, b=100.0)
    x1, x2 = x
    g1 = -2*(a - x1) - 4*b*x1*(x2 - x1^2)
    g2 =  2*b*(x2 - x1^2)
    return [g1, g2]
end

function hess_rosenbrock(x; a=1.0, b=100.0)
    x1, x2 = x
    h11 = 2 - 4*b*(x2 - x1^2) + 8*b*x1^2
    h12 = -4*b*x1
    h22 = 2*b
    return [h11 h12; h12 h22]
end

In [ ]:
xs = range(-2.0, 2.0; length=200)
ys = range(-1.0, 3.0; length=200)

Z = [rosenbrock([x, y]) for y in ys, x in xs];

In [ ]:
surface(
    xs, ys, Z;
    xlabel = L"x_1", ylabel = L"x_2", zlabel = L"f(x)",
    title  = "Rosenbrock function (3D surface)",
    legend = false,
    xlabelfontsize=14, ylabelfontsize=14, tickfontsize=10, legendfontsize=12
)


In [ ]:
contourf(
    xs, ys, log10.(Z),
    levels = 7, colorbar=true,
    xlabel = L"x_1", ylabel = L"x_2", title  = "Rosenbrock function (contour)",
    xlabelfontsize=14, ylabelfontsize=14, tickfontsize=10, legendfontsize=12, titlefontsize=14
)
# savefig("rosenbrock_contour.pdf")

In [ ]:
function steepest_descent(f, gradf, x0;
        α = 1e-3, maxit = 10_000, tol = 1e-8,
        xstar = nothing)

    x = copy(x0)

    g_hist = Float64[] # 勾配ノルムの履歴
    e_hist = (xstar === nothing) ? nothing : Float64[] 
      # 相対誤差の履歴（最適解が与えられた場合）

    for k in 1:maxit
        g = gradf(x) # 勾配計算
        ng = norm(g) # 勾配ノルム

        push!(g_hist, ng) # 履歴保存
        if e_hist !== nothing
            push!(e_hist, norm(x - xstar)/norm(xstar)) # 相対誤差の履歴保存
        end

        if ng < tol # 収束判定
            return x, k, g_hist, e_hist
        end

        x = x - α * g # 更新

    end
    return x, maxit, g_hist, e_hist
end

In [ ]:
function newton_method(f, gradf, hessf, x0;
        maxit = 100, tol = 1e-8,
        xstar = nothing)

    x = copy(x0)

    g_hist = Float64[] # 勾配ノルムの履歴
    e_hist = (xstar === nothing) ? nothing : Float64[] # 相対誤差の履歴（最適解が与えられた場合）

    for k in 1:maxit
        g = gradf(x) # 勾配計算
        ng = norm(g) # 勾配ノルム

        push!(g_hist, ng) # 履歴保存
        if e_hist !== nothing
            push!(e_hist, norm(x - xstar)/norm(xstar)) # 相対誤差の履歴保存
        end

        if ng < tol # 収束判定
            return x, k, g_hist, e_hist
        end

        H = hessf(x) # ヘッセ行列計算
        p = H \ g # 探索方向の計算
        x = x - p # 更新
    end

    return x, maxit, g_hist, e_hist
end

In [ ]:
x0 = [2.2; 2.2]
# x0 = [-1.2; 2.2]
x_opt = [1.0, 1.0]
x_sd, it_sd, g_sd, e_sd = 
    steepest_descent(rosenbrock, grad_rosenbrock, x0; α=1e-3, maxit = 10_000,xstar=x_opt)
x_nt, it_nt, g_nt, e_nt = 
    newton_method(rosenbrock, grad_rosenbrock, hess_rosenbrock, x0; xstar=x_opt)

In [ ]:
plot(0:length(g_sd)-1, g_sd; yscale=:log10, 
    linecolor=:black, linewidth=1.5,
    label="steepest descent", legend=:bottomright, 
    xlabel="iteration", ylabel=L"\Vert \nabla f(x^{(k)}) \Vert",
    xlabelfontsize=14, ylabelfontsize=14, tickfontsize=10, 
    legendfontsize=12, titlefontsize=14)
plot!(0:length(g_nt)-1, g_nt; yscale=:log10, label="Newton",
    linecolor=:black, linewidth=1.5,linestyle=:dashdot)
# savefig("rosenbrock_grad1.pdf")

In [ ]:
plot(0:9,g_sd[1:10]; yscale=:log10, 
    linecolor=:black, linewidth=1.5,
    label="steepest descent", legend=:bottomright, 
    xlabel="iteration", ylabel=L"\Vert \nabla f(x^{(k)}) \Vert",
    xlabelfontsize=14, ylabelfontsize=14, tickfontsize=10, 
    legendfontsize=12, titlefontsize=14)
plot!(0:length(g_nt)-1,g_nt; yscale=:log10, label="Newton",
    linecolor=:black, linewidth=1.5,linestyle=:dashdot)
# savefig("rosenbrock_grad2.pdf")

In [ ]:
plot(0:length(e_sd)-1,e_sd; yscale=:log10, 
    linecolor=:black, linewidth=1.5,
    label="steepest descent", legend=:bottomright, xlabel="iteration", ylabel=L"\Vert x^{(k)} - x^* \Vert / \Vert x^* \Vert",
    xlabelfontsize=14, ylabelfontsize=14, tickfontsize=10, legendfontsize=12, titlefontsize=14)
plot!(0:length(e_nt)-1,e_nt; yscale=:log10, label="Newton",
    linecolor=:black, linewidth=1.5,linestyle=:dashdot)
# savefig("rosenbrock_error1.pdf")

In [ ]:
plot(0:9,e_sd[1:10]; yscale=:log10, 
    linecolor=:black, linewidth=1.5,
    label="steepest descent", legend=:bottomright, xlabel="iteration", ylabel=L"\Vert x^{(k)} - x^* \Vert / \Vert x^* \Vert",
    xlabelfontsize=14, ylabelfontsize=14, tickfontsize=10, legendfontsize=12, titlefontsize=14)
plot!(0:length(e_nt)-1, e_nt; yscale=:log10, label="Newton",
    linecolor=:black, linewidth=1.5,linestyle=:dashdot)
# savefig("rosenbrock_error2.pdf")

# Optimパッケージの使用例

In [ ]:
using Optim

f(x) = rosenbrock(x)
g(x) = grad_rosenbrock(x)
h(x) = hess_rosenbrock(x)     

x0 = [2.2, 2.2]

# ニュートン法
res = optimize(f, g, h, x0, Newton(), inplace = false, Optim.Options(
    iterations = 100, # 最大反復回数
    g_abstol = 1e-8, # 勾配ノルムの許容誤差
    x_abstol = 0.0, # 反復間の変化量の許容誤差
    f_abstol = 0.0  # 関数値の変化量の許容誤差
)) 

In [ ]:
@show Optim.minimizer(res) Optim.minimum(res) Optim.iterations(res)

In [ ]:
g!(G, x) = (G .= grad_rosenbrock(x))  
h!(H, x) = (H .= hess_rosenbrock(x))    

res = optimize(f, g!, h!, x0, Newton(), Optim.Options(
    iterations = 1_000, # 最大反復回数
    g_abstol = 1e-8, # 勾配ノルムの許容誤差
    x_abstol = 0.0, # 反復間の変化量の許容誤差
    f_abstol = 0.0  # 関数値の変化量の許容誤差
)) 

In [ ]:
res = optimize(f, x0, NelderMead()) 

In [ ]:
res = optimize(f, g!, x0, GradientDescent()) 